# Demo 2 — Polymarket panel experiments

This notebook runs the five panel experiments implemented in `src/polymarket/panel_experiments.py` on the checked-in `polymarket_daily_panel_60plus.csv` dataset.

The experiments correspond to the documents in `docs/polymarket_panel/`:

1. Persistent predictability net of costs.
2. Liquidity and forecastability.
3. Stable cross-sectional factors.
4. Unresolved open-market risk.
5. Validation design stress test.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("Could not find repo root containing src/ and data/.")


ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUT = ROOT / "Outputs" / "demo_polymarket_panel_experiments"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "reports").mkdir(exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Repo root:", ROOT)
print("Output directory:", OUT)

## Load panel and build features

The daily panel loader enforces typed dates/numerics and sorts by `(market_id, date)`. Feature construction then adds lagged returns, rolling momentum/volatility, maturity, liquidity, illiquidity, and cross-sectional rank/mispricing features.

In [ ]:
from polymarket.panel_experiments import (
    add_features,
    load_panel,
    make_split,
    run_exp_01,
    run_exp_02,
    run_exp_03,
    run_exp_04,
    run_exp_05,
    sanity_check_no_overlap,
    split_60_20_20,
    to_markdown,
)

PANEL_CSV = ROOT / "data" / "prediction_market" / "polymarket_daily_panel_60plus.csv"
assert PANEL_CSV.is_file(), f"Missing {PANEL_CSV}"

panel_raw = load_panel(PANEL_CSV)
panel = add_features(panel_raw)

summary = {
    "rows": len(panel),
    "markets": int(panel["market_id"].nunique()),
    "dates": int(panel["date"].nunique()),
    "date_min": str(panel["date"].min().date()),
    "date_max": str(panel["date"].max().date()),
    "columns_after_features": len(panel.columns),
}
display(pd.DataFrame([summary]))
display(panel.head(3))

In [ ]:
split = split_60_20_20(panel)
train, valid, test = make_split(panel, split)

split_table = pd.DataFrame(
    [
        {"split": "train", "rows": len(train), "dates": train["date"].nunique(), "start": train["date"].min(), "end": train["date"].max()},
        {"split": "validation", "rows": len(valid), "dates": valid["date"].nunique(), "start": valid["date"].min(), "end": valid["date"].max()},
        {"split": "test", "rows": len(test), "dates": test["date"].nunique(), "start": test["date"].min(), "end": test["date"].max()},
    ]
)
display(split_table)
display(pd.DataFrame([sanity_check_no_overlap(train, valid, test)]))

## Run the five experiment functions

Each runner returns a metrics dictionary, a sanity-check dictionary, and paths to generated artifacts. The notebook writes the same markdown-style reports as the CLI, but under `Outputs/demo_polymarket_panel_experiments/` so it does not overwrite the checked-in result set.

In [ ]:
experiment_runs = [
    ("Experiment 1: Persistent Predictability", run_exp_01),
    ("Experiment 2: Liquidity and Forecastability", run_exp_02),
    ("Experiment 3: Cross-Sectional Factors", run_exp_03),
    ("Experiment 4: Unresolved Market Risk", run_exp_04),
    ("Experiment 5: Validation Design", run_exp_05),
]

results = {}
summary_lines = ["# Demo Panel Experiment Results", "", "## Experiment index", ""]
for i, (name, fn) in enumerate(experiment_runs, start=1):
    out_i = OUT / f"exp_{i:02d}"
    out_i.mkdir(parents=True, exist_ok=True)
    result = fn(panel, out_i)
    results[name] = result

    report_path = OUT / "reports" / f"exp_{i:02d}_report.md"
    to_markdown(name, result, report_path)
    failed = [k for k, v in result["checks"].items() if not v]
    status = "PASS" if not failed else f"FAIL ({len(failed)} checks)"
    summary_lines.append(f"- [{name}](./reports/{report_path.name})")
    summary_lines.append(f"  - Status: {status}")

(OUT / "master_report.md").write_text("\n".join(summary_lines), encoding="utf-8")
print("Wrote:", OUT / "master_report.md")

In [ ]:
metric_rows = []
check_rows = []
for name, result in results.items():
    short = name.split(":", 1)[0].replace("Experiment ", "Exp ")
    row = {"experiment": short}
    row.update(result["metrics"])
    metric_rows.append(row)
    for check, ok in result["checks"].items():
        check_rows.append({"experiment": short, "check": check, "pass": bool(ok)})

metrics_df = pd.DataFrame(metric_rows)
checks_df = pd.DataFrame(check_rows)

display(metrics_df.round(6))
print("All sanity checks passed:", bool(checks_df["pass"].all()))
display(checks_df.groupby("experiment")["pass"].agg(["count", "sum"]).rename(columns={"count": "checks", "sum": "passed"}))

## Key output artifacts

The next cells load a few representative generated tables and figures. These are the same artifacts referenced by the per-experiment reports.

In [ ]:
# Experiment 1: decile returns and cumulative PnL figure.
exp1_deciles = pd.read_csv(OUT / "exp_01" / "table_exp01_decile_returns.csv")
display(exp1_deciles)
display(Image(filename=str(OUT / "exp_01" / "fig_exp01_cum_pnl.png")))

In [ ]:
# Experiment 2: signal quality by train-defined liquidity bucket.
exp2_buckets = pd.read_csv(OUT / "exp_02" / "table_exp02_bucket_metrics.csv")
display(exp2_buckets.round(6))
display(Image(filename=str(OUT / "exp_02" / "fig_exp02_rankic_bucket.png")))

In [ ]:
# Experiment 3 and 4 compact tables.
exp3 = pd.read_csv(OUT / "exp_03" / "table_exp03_incremental.csv")
exp4 = pd.read_csv(OUT / "exp_04" / "table_exp04_reliability.csv")
display(exp3.round(6))
display(exp4.round(6))

In [ ]:
# Experiment 5: validation-vs-test splitter comparison.
exp5 = pd.read_csv(OUT / "exp_05" / "table_exp05_splitter_compare.csv")
display(exp5.round(6))
display(Image(filename=str(OUT / "exp_05" / "fig_exp05_val_vs_test.png")))

## Reading the demo results

The important pattern is not a single metric. The suite shows how prediction, liquidity heterogeneity, cross-sectional factors, unresolved-risk haircuts, and validation design interact. A robust deployment should prioritize leakage-safe split governance and ranked/cost-aware signal evaluation over raw point-forecast fit.